# IPL Data Cleaning and EDA

This notebook shows how to clean IPL CSV files, inspect the structure, and prepare them for feature engineering and model training.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

from utils.preprocessing import build_demo_data, load_ipl_data, standardize_match_schema, standardize_delivery_schema
from utils.feature_engineering import engineer_ball_by_ball_features, build_training_frame

## 1. Load Raw Data

If real CSV files are available, the notebook loads them. Otherwise it creates a small demo dataset so you can still learn the workflow.

In [ ]:
data_dir = Path('..') / 'data'
matches_path = data_dir / 'matches.csv'
deliveries_path = data_dir / 'deliveries.csv'

if matches_path.exists() and deliveries_path.exists():
    matches, deliveries = load_ipl_data(matches_path, deliveries_path)
else:
    matches, deliveries = build_demo_data()

matches.head(), deliveries.head()

## 2. Clean Column Names

Different IPL datasets use slightly different column names. Standardizing them makes the rest of the project simpler.

In [ ]:
matches = standardize_match_schema(matches)
deliveries = standardize_delivery_schema(deliveries)

print(matches.columns.tolist())
print(deliveries.columns.tolist())

## 3. Check Missing Values and Shapes

This helps you spot obvious issues before modeling.

In [ ]:
print('Matches shape:', matches.shape)
print('Deliveries shape:', deliveries.shape)
print('
Missing values in matches:')
print(matches.isna().sum().sort_values(ascending=False).head(10))
print('
Missing values in deliveries:')
print(deliveries.isna().sum().sort_values(ascending=False).head(10))

## 4. Create Cricket Features

Now the raw rows are converted into prediction features like run rate, wickets remaining, and pressure index.

In [ ]:
engineered = engineer_ball_by_ball_features(deliveries, matches)
training_frame = build_training_frame(matches, deliveries)
engineered.head()

## 5. Simple Visual Check

A score progression chart gives a quick sense of how the chase moved ball by ball.

In [ ]:
sample_match_id = engineered['match_id'].iloc[0]
sample = engineered[engineered['match_id'] == sample_match_id].copy()
sample['ball_number'] = range(1, len(sample) + 1)
sample['innings_runs'] = sample['total_runs'].cumsum()
px.line(sample, x='ball_number', y='innings_runs', title='Run Progression')